# Tutorial: Despacho de Ambulâncias com `FilterStore`

**Público:** alunos que já conhecem `Store` e precisam selecionar objetos por tipo.

**Pré-requisitos:** entender a ideia de objeto identificável na simulação.

**Objetivos de aprendizagem:**

- usar `FilterStore` para buscar veículos compatíveis;
- separar a triagem da chamada da alocação da frota;
- entender por que este caso não é bem representado por `Resource` simples.


## Roteiro

1. Entender o cenário da central médica.
2. Separar central e frota em estruturas diferentes.
3. Modelar a busca por veículo compatível.
4. Rodar a simulação.
5. Interpretar a espera por tipo de ambulância.


In [1]:
from __future__ import annotations

import simpy

print(f"Versão do SimPy: {simpy.__version__}")

Versão do SimPy: 4.1.1


## 1. Cenário

Uma chamada entra na central, é qualificada por um regulador e depois precisa de uma ambulância compatível com o tipo do caso.

Temos dois tipos neste exemplo:

- `BASICA`
- `UTI`

O ponto central é que não basta liberar "uma ambulância qualquer". Precisamos liberar **um veículo do tipo correto**.


## 2. Conceito fundamental: por que `FilterStore`?

`FilterStore` é apropriado quando temos uma coleção de objetos e queremos recuperar apenas os que satisfazem uma condição.

Aqui a condição é:

`item["tipo"] == tipo solicitado`

Essa é a diferença chave em relação ao `Store` simples.


In [2]:
CHAMADAS = [
    ("C001", 0.0, "BASICA", 6.0),
    ("C002", 1.0, "BASICA", 5.0),
    ("C003", 2.0, "UTI", 8.0),
    ("C004", 4.0, "BASICA", 4.0),
]

CHAMADAS

[('C001', 0.0, 'BASICA', 6.0),
 ('C002', 1.0, 'BASICA', 5.0),
 ('C003', 2.0, 'UTI', 8.0),
 ('C004', 4.0, 'BASICA', 4.0)]

### Estrutura dos dados

Cada chamada contém:

- código;
- chegada;
- tipo de ambulância necessário;
- tempo total fora da base.


In [3]:
def chamada(env, codigo, chegada, tipo, ciclo, central, frota):
    yield env.timeout(chegada)
    print(f"{env.now:04.1f} h | {codigo} entra na central | tipo={tipo}")

    with central.request() as req:
        yield req
        yield env.timeout(0.2)

    ambulancia = yield frota.get(filter=lambda item: item["tipo"] == tipo)
    print(f"{env.now:04.1f} h | {codigo} despachada com {ambulancia['id']}")
    yield env.timeout(ciclo)
    yield frota.put(ambulancia)
    print(f"{env.now:04.1f} h | {ambulancia['id']} retorna à base")

## 3. Onde está a lógica principal?

Ela está nesta linha:

`frota.get(filter=lambda item: item["tipo"] == tipo)`

Leia assim:

> "me entregue um veículo disponível cujo tipo seja compatível com a chamada".

Isso é exatamente o tipo de pergunta que `FilterStore` resolve muito bem.


In [4]:
def executar_simulacao():
    env = simpy.Environment()
    central = simpy.Resource(env, capacity=1)
    frota = simpy.FilterStore(env, capacity=10)
    frota.items.extend(
        [
            {"id": "AMB-1", "tipo": "BASICA"},
            {"id": "UTI-1", "tipo": "UTI"},
        ]
    )

    for dados in CHAMADAS:
        env.process(chamada(env, *dados, central, frota))

    env.run()


executar_simulacao()

00.0 h | C001 entra na central | tipo=BASICA
00.2 h | C001 despachada com AMB-1
01.0 h | C002 entra na central | tipo=BASICA
02.0 h | C003 entra na central | tipo=UTI
02.2 h | C003 despachada com UTI-1
04.0 h | C004 entra na central | tipo=BASICA
06.2 h | AMB-1 retorna à base
06.2 h | C002 despachada com AMB-1
10.2 h | UTI-1 retorna à base
11.2 h | AMB-1 retorna à base
11.2 h | C004 despachada com AMB-1
15.2 h | AMB-1 retorna à base


## 4. O que observar na saída

Pontos principais:

- `C001` usa a ambulância básica logo no início;
- `C002` e `C004` precisam esperar a volta do mesmo veículo básico;
- `C003` não compete com eles porque usa a ambulância UTI.

Aprendizado central:

**a espera depende do tipo de recurso requerido, não apenas do volume total de chamadas**.


## 5. Erro comum

Um erro comum é representar a frota inteira como um único `Resource` com capacidade `2`.

Isso perderia a diferença entre ambulância básica e UTI móvel, que é justamente a informação mais importante do problema.


## 6. Exercícios

1. Adicione mais uma ambulância básica e compare a fila de `C002` e `C004`.
2. Troque `C003` para tipo `BASICA`. O que muda?
3. Explique por que a central e a frota foram modeladas separadamente.


In [5]:
# Espaço para experimentos:
# - altere CHAMADAS;
# - altere a lista inicial da frota;
# - rode novamente a simulação.

## 7. Extensão sugerida

Uma evolução natural deste modelo é incluir:

- múltiplas bases;
- geografia;
- tempo de deslocamento;
- prioridade clínica;
- política de despacho ótima.

Aí o problema começa a encostar em localização e roteamento, não apenas em fila discreta.
